# Evaluate + Push — Qwen2.5-Coder-7B EduCode Rwanda v2

**Purpose:** Load the trained adapter from the saved dataset, run evaluation (perplexity + qualitative), then push to HuggingFace Hub.

**Required inputs (add in Kaggle sidebar):**
- `EducodeTrainedModel` — your saved adapter dataset
- `educode-v2-training-data` — for validation perplexity

**Required secret:** `HF_TOKEN` in Kaggle Settings → Secrets

**Accelerator:** GPU T4 x1 (needed for model inference)

In [ ]:
%%capture
!pip install -q -U bitsandbytes
!pip install -q -U transformers==4.46.3 peft==0.13.2 accelerate==1.1.1 datasets==3.1.0 huggingface_hub

In [ ]:
import os, torch
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory // 1024**3} GB")

## 1. Load base model + trained adapter

In [ ]:
import glob
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"

# Auto-detect the dataset mount path (Kaggle mounts differ by how dataset was saved)
_candidates = [
    "/kaggle/input/educodetrainedmodel/qwen-7b-educode-v2",
    "/kaggle/input/datasets/mitalibela/educodetrainedmodel/qwen-7b-educode-v2",
]
DATASET_DIR = next((p for p in _candidates if os.path.isdir(p)), None)
if DATASET_DIR is None:
    raise FileNotFoundError(
        "Cannot find EducodeTrainedModel dataset. "
        "Did you add it as an input in the Kaggle sidebar?\n"
        f"Tried: {_candidates}"
    )
print(f"Dataset found at: {DATASET_DIR}")

# adapter_config.json is at root if final save ran, otherwise in the latest checkpoint
if os.path.exists(os.path.join(DATASET_DIR, "adapter_config.json")):
    ADAPTER_PATH = DATASET_DIR
else:
    checkpoints = sorted(glob.glob(os.path.join(DATASET_DIR, "checkpoint-*")))
    if not checkpoints:
        raise FileNotFoundError(f"No adapter_config.json or checkpoints found in {DATASET_DIR}")
    ADAPTER_PATH = checkpoints[-1]
print(f"Adapter path:   {ADAPTER_PATH}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading base model in 4-bit...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True,
)

print("Applying LoRA adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print(f"\nReady. Device: {next(model.parameters()).device}")

## 2. Load validation dataset

In [ ]:
from datasets import load_dataset

DATA_DIR = "/kaggle/input/educode-v2-training-data"

dataset = load_dataset(
    "json",
    data_files={"validation": f"{DATA_DIR}/val_merged.jsonl"},
)

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(format_chat, num_proc=2)
print(f"Validation examples: {len(dataset['validation'])}")

## 3. Perplexity Evaluation

**Perplexity** measures how confidently the model predicts the validation set — lower is better.
A random model over Qwen's 152,064-token vocabulary scores ~152,000.
A well-fine-tuned model on this domain should score below 5.

In [ ]:
import math

EVAL_SAMPLES = 80
EVAL_MAX_LEN = 512   # 512 not 1024 — avoids OOM without gradient checkpointing

print(f"Computing perplexity on {EVAL_SAMPLES} validation examples (max_length={EVAL_MAX_LEN})...")
model.eval()
total_loss, n = 0.0, 0

with torch.no_grad():
    for example in dataset['validation'].select(range(EVAL_SAMPLES)):
        inputs = tokenizer(
            example['text'], return_tensors='pt',
            truncation=True, max_length=EVAL_MAX_LEN,
        ).to('cuda:0')
        if inputs['input_ids'].shape[1] < 10:
            continue
        outputs = model(**inputs, labels=inputs['input_ids'])
        total_loss += outputs.loss.item()
        n += 1

avg_loss   = total_loss / n
perplexity = math.exp(avg_loss)

print(f"\n{'='*50}")
print(f"  Avg validation loss:   {avg_loss:.4f}")
print(f"  Validation perplexity: {perplexity:.2f}")
print(f"{'='*50}")
print(f"\n  Random baseline (152k vocab): ~152,000")
print(f"  Target for this domain:       < 5")

## 4. Qualitative Evaluation — Sample Model Responses

Four representative student scenarios covering error diagnosis, concept questions, and code review.

In [ ]:
TEST_CASES = [
    {
        "label": "const reassignment",
        "message": (
            "This code has an error. Explain what is wrong and how to fix it:\n"
            "```javascript\nconst score = 0;\nscore = score + 10;\nconsole.log(score);\n```\n"
            "Error: TypeError: Assignment to constant variable."
        ),
    },
    {
        "label": "undefined variable (TDZ)",
        "message": (
            "This code has an error. Explain what is wrong and how to fix it:\n"
            "```javascript\nconsole.log(name);\nlet name = 'Alice';\n```\n"
            "Error: ReferenceError: Cannot access 'name' before initialization"
        ),
    },
    {
        "label": "missing return value",
        "message": (
            "This code has an error. Explain what is wrong and how to fix it:\n"
            "```javascript\nfunction add(a, b) {\n    let result = a + b;\n}\nconsole.log(add(3, 4));\n```\n"
            "Output is: undefined"
        ),
    },
    {
        "label": "concept — arrow functions vs regular",
        "message": "What is the difference between a regular function and an arrow function in JavaScript?",
    },
]

EVAL_MAX_LEN = 512

for tc in TEST_CASES:
    messages   = [{"role": "user", "content": tc["message"]}]
    prompt_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs     = tokenizer(prompt_str, return_tensors="pt", truncation=True, max_length=EVAL_MAX_LEN).to("cuda:0")

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.3,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    print(f"\n{'─'*65}")
    print(f"SCENARIO: {tc['label']}")
    print(f"INPUT:    {tc['message'][:120]}{'...' if len(tc['message'])>120 else ''}")
    print(f"\nMWARIMU:\n{response}")

## 5. Push adapter to HuggingFace Hub

Add `HF_TOKEN` to Kaggle Settings → Secrets before running this cell.

In [ ]:
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found. Add it in Kaggle Settings → Secrets.")

login(token=HF_TOKEN)

HF_REPO = "DavBelaa/qwen25-coder-7b-educode-rwanda-v2"

print(f"Pushing adapter to {HF_REPO}...")
model.push_to_hub(HF_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
print(f"\nDone! https://huggingface.co/{HF_REPO}")